# Phase 2 Evaluation-Complete Notebook

This notebook keeps the original current-frame proxy result as a diagnostic agreement check, then runs the fairer future pseudo-label IoU evaluation.

Important: all metrics are **bounded projected-depth semantic occupancy** metrics from sparse LiDAR-converted depth and front semantic segmentation. They are not dense simulator 3D ground-truth IoU.


## 1. Setup


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Image, Markdown, display

from src.evaluation.phase2_carla_eval import (
    FuturePseudoEvalConfig,
    Phase2CarlaEvalConfig,
    iter_shadow_records,
    run_bounded_carla_iou_evaluation,
    run_future_pseudo_label_evaluation,
)

MAX_FRAMES = 200
ANCHOR_FRAMES = 200
OUTPUT_DIR = ROOT / "outputs" / "phase2"
OCCUPANCY_THRESHOLD = 0.25
DISAGREEMENT_THRESHOLD = 0.05

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Current-Frame Proxy Agreement Diagnostic

This is not final IoU. Baseline is expected to score near-perfectly because the target is the current frame itself. Use this cell only to confirm data loading, segmentation decoding, occupancy construction, temporal fusion changes, and shadow-mode logging.


In [ ]:
current_config = Phase2CarlaEvalConfig(
    max_frames=MAX_FRAMES,
    output_dir=OUTPUT_DIR,
    occupancy_threshold=OCCUPANCY_THRESHOLD,
    disagreement_threshold=DISAGREEMENT_THRESHOLD,
    bev_frame_count=5,
)

current_artifacts = run_bounded_carla_iou_evaluation(config=current_config)
for name, path in current_artifacts.__dict__.items():
    print(f"{name}: {path}")


## 3. Future Pseudo-Label IoU Evaluation

For each anchor frame `t`, baseline uses frame `t`, temporal fusion uses frames `t..t-4`, and the pseudo-label target is the union of frames `t..t+4` transformed into frame `t`. This makes the target denser than one sparse LiDAR frame and avoids comparing the baseline to itself.


In [ ]:
future_config = FuturePseudoEvalConfig(
    anchor_count=ANCHOR_FRAMES,
    output_dir=OUTPUT_DIR,
    occupancy_threshold=OCCUPANCY_THRESHOLD,
    disagreement_threshold=DISAGREEMENT_THRESHOLD,
    bev_frame_count=5,
    alignment_frame_count=3,
)

future_artifacts = run_future_pseudo_label_evaluation(config=future_config)
for name, path in future_artifacts.__dict__.items():
    print(f"{name}: {path}")


## 4. Future Pseudo-Label IoU Table


In [ ]:
def load_summary(path: Path) -> dict:
    return json.loads(Path(path).read_text(encoding="utf-8"))

baseline = load_summary(future_artifacts.baseline_iou)
temporal = load_summary(future_artifacts.temporal_iou)
summary = load_summary(future_artifacts.eval_summary)

base_by_id = {row["class_id"]: row for row in baseline["classes"]}
temp_by_id = {row["class_id"]: row for row in temporal["classes"]}
rows = []
for class_id, base in base_by_id.items():
    temp = temp_by_id[class_id]
    informative = base["union"] > 0 or temp["union"] > 0
    rows.append(
        {
            "class_name": base["class_name"],
            "baseline_iou": base["iou"],
            "temporal_iou": temp["iou"],
            "delta": temp["iou"] - base["iou"],
            "baseline_union": base["union"],
            "temporal_union": temp["union"],
            "informative": informative,
        }
    )

df = pd.DataFrame(rows)
display(df.style.format({"baseline_iou": "{:.3f}", "temporal_iou": "{:.3f}", "delta": "{:+.3f}"}))
print(f"Processed anchors: {summary['processed_anchor_count']} / requested {summary['requested_anchor_count']}")
print(f"Selected fusion profile: {summary['selected_profile']} @ threshold {summary['selected_occupancy_threshold']}")


## 5. Threshold Ablation


In [ ]:
ablation = load_summary(future_artifacts.ablation_summary)
ablation_rows = []
for row in ablation["rows"]:
    flat = {"profile": row["profile"], "threshold": row["occupancy_threshold"]}
    for class_id, class_row in row["classes"].items():
        name = class_row["class_name"]
        flat[f"{name}_iou"] = class_row["iou"]
        flat[f"{name}_union"] = class_row["union"]
    ablation_rows.append(flat)

display(pd.DataFrame(ablation_rows).sort_values(["vehicle_iou", "road_iou"], ascending=False))
print("Selected:", ablation["selected_profile"], ablation["selected_occupancy_threshold"], ablation["selected_weights"])


## 6. Shadow-Mode Summary


In [ ]:
records = list(iter_shadow_records(future_artifacts.shadow_records))
clusters = load_summary(future_artifacts.shadow_clusters)
if records:
    disagreements = [record["disagreement_rate"] for record in records]
    print(f"Evaluated anchors: {len(records)}")
    print(f"Flagged frames: {sum(record['flagged'] for record in records)}")
    print(f"Mean disagreement: {sum(disagreements) / len(disagreements):.4f}")
    print(f"Max disagreement: {max(disagreements):.4f}")
print(json.dumps(clusters, indent=2))


## 7. Alignment Diagnostics

Inspect these before trusting future pseudo-label IoU. Adjacent frames transformed into the anchor frame should be spatially close, not mirrored or wildly shifted.


In [ ]:
alignment = load_summary(future_artifacts.alignment_summary)
print(json.dumps({k: v for k, v in alignment.items() if k != "records"}, indent=2))
for image_path in alignment["image_paths"]:
    display(Image(filename=image_path))


## 8. Visual Checks


In [ ]:
for image_path in future_artifacts.bev_images[:5]:
    display(Image(filename=str(image_path)))


## 9. Interpretation Notes


In [ ]:
notes = []
for row in rows:
    if not row["informative"]:
        notes.append(f"{row['class_name']}: non-informative empty union in this bounded run.")
    elif row["delta"] >= 0:
        notes.append(f"{row['class_name']}: temporal fusion improved/preserved future-pseudo IoU by {row['delta']:+.3f}.")
    else:
        notes.append(f"{row['class_name']}: temporal fusion reduced future-pseudo IoU by {row['delta']:+.3f}; inspect alignment and ablation images.")
notes.append("These are pseudo-label metrics from sparse projected LiDAR, not true dense 3D occupancy ground truth.")
display(Markdown("\n".join(f"- {note}" for note in notes)))
